In [1]:
# %pip install langchain langchain-core langchain-community pypdf pymupdf sentence-transformers chromadb

In [2]:
from langchain_core.documents import Document

In [3]:
sample_doc = Document(
    page_content="Hello World",
    metadata={"source": "https:/www.google.com"}
)

In [4]:
sample_doc

Document(metadata={'source': 'https:/www.google.com'}, page_content='Hello World')

In [5]:
type(sample_doc)

langchain_core.documents.base.Document

##### Load text and pdf data

In [6]:
# Text data
# from langchain_community.document_loaders import TextLoader

# loader = TextLoader("data/Python.txt", encoding="utf-8")

# document = loader.load()
# document

In [7]:
# PDF data
# from langchain_community.document_loaders import PyPDFLoader

# pdf_loader = PyPDFLoader("data/research.pdf")

# document = pdf_loader.load()
# document

In [8]:
# PDF data
# from langchain_community.document_loaders import PyMuPDFLoader

# pdf_loader = PyMuPDFLoader("data/research.pdf")

# document = pdf_loader.load()
# document

### Ingestion Pipeline

#### Document

In [6]:
# Data => Document
import os
from langchain_community.document_loaders.pdf import PyPDFLoader

C:\Users\user\AppData\Local\Temp\ipykernel_18176\1162334499.py:3: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders.pdf import PyPDFLoader


In [7]:
def load_all_pdfs():
    folder_path = "data/pdfs"
    num_docs = 0
    all_docs = []
    
    for filename in os.listdir(folder_path):
        if filename.lower().endswith(".pdf"):
            # complete file path
            pdf_path = os.path.join(folder_path, filename)
            
            pdf_loader = PyPDFLoader(pdf_path)
            doc = pdf_loader.load()
            
            all_docs.extend(doc)
            num_docs += 1
            
    print("total pdfs:", num_docs)
    print("total pages", len(all_docs))
    
    return all_docs

In [8]:
all_pdf_documents = load_all_pdfs()

total pdfs: 2
total pages 36


In [9]:
type(all_pdf_documents[1])

langchain_core.documents.base.Document

#### Chunks

In [13]:
# chunks
# %pip install langchain_text_splitters 

In [10]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

# chunk_size = maximum characters in each chunk, chunk_overlap = overlap of characters between 2 chunks
def split_docs(documents, chunk_size=500, chunk_overlap=50):
    
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size = chunk_size,
        chunk_overlap = chunk_overlap
    )
    
    chunked_docs = text_splitter.split_documents(documents)
    return chunked_docs # returns a list

In [11]:
chunks = split_docs(all_pdf_documents)

In [12]:
len(chunks) # Each chunk is a document

337

#### Embedding

In [13]:
from sentence_transformers import SentenceTransformer

In [14]:
class EmbeddingManager:
    def __init__(self, model_name = "all-MiniLM-L6-v2"):
        
        self.model_name = model_name
        print("loading model....", self.model_name)
        
        self.model = SentenceTransformer(self.model_name)
        print("embedding dimensions=", self.model.get_sentence_embedding_dimension())
        
    def get_embeddings(self, text):
        embeddings = self.model.encode(text, show_progress_bar=True)
        print("embedding shape", embeddings.shape)
        return embeddings

In [15]:
embedding_manager = EmbeddingManager()

loading model.... all-MiniLM-L6-v2
embedding dimensions= 384


C:\Users\user\AppData\Local\Temp\ipykernel_18176\1102271777.py:8: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  print("embedding dimensions=", self.model.get_sentence_embedding_dimension())


#### Vector Store

In [16]:
import chromadb
import uuid  # creates indexes for our individual documents

In [ ]:
class VectorStoreManager:
    def __init__(self, persist_directory="data/vector_store", collection_name="pdf_documents"):
        self.collection_name = collection_name
        self.persist_directory = persist_directory # hard disk path where vector store is located
        self.collection = None
        self.client = None  # helps connecting others with vector store
        
        self._initialize_store()
        
        
    def _initialize_store(self):
        os.makedirs(self.persist_directory, exist_ok=True) # if vector store does not exist, create it at persist directory path
        
        # create a client
        self.client = chromadb.PersistentClient(path=self.persist_directory)
        
        # create the collection
        self.collection = self.client.get_or_create_collection(
            name=self.collection_name,
            metadata={"description":"vector store collection for pdf embeddings in RAG"}
        )
        
        print("initialized vector store with collection:", self.collection_name)
        print("docs in collection:", self.collection.count())
        
        
    # documents = all chunks, embeddings = all embeddings of the chunks
    def add_documents(self, documents, embeddings):
        if(len(documents)) != len(embeddings):
            raise ValueError("num of documents does not match num of embeddings")
        
        
        # store in collection => id, embedding, document, metadata
        ids = []
        all_metadata = []
        documents_content = []
        embeddings_list = []
        
        # for each chunk and their embedding => add to collection
        for i, (doc, embedding) in enumerate(zip(documents, embeddings)): 
            doc_id = f"doc_{uuid.uuid4()}" # generates random id for each doc
            ids.append(doc_id)
            
            metadata = dict(doc.metadata)
            metadata["doc_index"] = i
            metadata["content_length"] = len(doc.page_content)
            all_metadata.append(metadata)
            
            documents_content.append(doc.page_content)
            
            embeddings_list.append(embedding.tolist())
            
            self.collection.add(
                ids=ids,
                metadatas=all_metadata,
                documents=documents_content,
                embeddings=embeddings_list
            )
            
        print("total documents added in vector store:", len(documents_content))
        print("docs in collection:", self.collection.count())

In [18]:
vector_store = VectorStoreManager()

initialized vector store with collection: pdf_documents
docs in collection: 337


In [ ]:
# data => documents => chunks => embeddings => store in vector store

texts = [doc.page_content for doc in chunks]

embeddings = embedding_manager.get_embeddings(texts)

vector_store.add_documents(chunks, embeddings)

Batches:   0%|          | 0/11 [00:00<?, ?it/s]

embedding shape (337, 384)
total documents added in vector store: 337
docs in collection: 337


### Retrieval Pipeline

In [19]:
from sklearn.metrics.pairwise import cosine_similarity

In [21]:
class RAGRetriever:
    def __init__(self, embedding_manager, vector_store):
        self.embedding_manager = embedding_manager
        self.vector_store = vector_store
        
        
    def retrieve(self, query, top_k=5, score_threshold=0.0):  # score_threshold => minimum cosine similarity to consider for results
        # query => embedding
        query_embeddings = self.embedding_manager.get_embeddings([query])[0] # query is passed as a , [0] index refers to 1st query, because chromadb supports multiple query at once
        
        # semantic search
        results = self.vector_store.collection.query(  # query from vector store using chromadb's internal cosine similarity
            query_embeddings = [query_embeddings.tolist()],
            n_results = top_k
        )
        
        # cosine similarity
        retrived_docs = []
        if results["documents"] and results["documents"][0]: # for a valid document (not empty)
            ids = results["ids"][0]
            metadatas = results["metadatas"][0]
            documents = results["documents"][0]
            distances = results["distances"][0] 
        
        
            for i, (doc_id, metadata, document, distance) in enumerate(zip(ids, metadatas, documents, distances)):
                similarity_score = 1 - distance # because chromadb returns cosine distance, not similarity
                
                if similarity_score >= score_threshold:
                    retrived_docs.append({
                        "id": doc_id,
                        "document": document,
                        "metadata": metadata,
                        "similarity_score": similarity_score,
                        "rank": i + 1
                    })
        
            
        else:
            print("No documents found")
            
        return retrived_docs

In [26]:
rag_retriever = RAGRetriever(embedding_manager, vector_store)

In [27]:
rag_retriever.retrieve("What is encoder decoder?")  # context

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

embedding shape (1, 384)


[{'id': 'doc_b7c35cc6-8cfe-416f-8282-f0ed53e356dc',
  'document': 'Decoder: The decoder is also composed of a stack of N = 6identical layers. In addition to the two\nsub-layers in each encoder layer, the decoder inserts a third sub-layer, which performs multi-head\nattention over the output of the encoder stack. Similar to the encoder, we employ residual connections\naround each of the sub-layers, followed by layer normalization. We also modify the self-attention\nsub-layer in the decoder stack to prevent positions from attending to subsequent positions. This',
  'metadata': {'moddate': '2024-04-10T21:11:43+00:00',
   'creationdate': '2024-04-10T21:11:43+00:00',
   'creator': 'LaTeX with hyperref',
   'page': 2,
   'page_label': '3',
   'trapped': '/False',
   'source': 'data/pdfs\\research1.pdf',
   'author': '',
   'content_length': 494,
   'keywords': '',
   'title': '',
   'producer': 'pdfTeX-1.40.25',
   'doc_index': 20,
   'subject': '',
   'ptex.fullbanner': 'This is pdfTeX, Ver

### Integrate with LLMs

#### OpenAI - GPT

In [36]:
from dotenv import load_dotenv

load_dotenv()

API_KEY_OPENAI = os.getenv("OPENAI_API_KEY")

In [ ]:
# %pip install langchain-openai

In [ ]:
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(
    openai_api_key=API_KEY_OPENAI,
    model="gpt-5.5",
    temperature=0.1,  # low creativity - fact based answer
    max_tokens=1024   # maximum tokens for generated response
)

In [31]:
# generate our retrieval-augmented output
def generate_output(query, retriever, llm, top_k=3):
    results = retriever.retrieve(query, top_k)
    
    context = "\n".join(doc["document"] for doc in results) if results else ""
    
    if not context:
        print("We found no relevant context for the given query")
        
        
    # prompt = context + query
    prompt = f""" use given context to generate the answer
                Context: {context}
                Query: {query} """
                
    response = llm.invoke(prompt) # gpt expects a string as prompt
    return response.content

In [ ]:
answer = generate_output("What is RAG?", rag_retriever, llm)

In [ ]:
print(answer) 

#### Groq

In [38]:
from dotenv import load_dotenv

load_dotenv()

API_KEY_GROQ = os.getenv("GROQ_API_KEY")

In [ ]:
# %pip install langchain-groq

In [48]:
from langchain_groq import ChatGroq

llm = ChatGroq(
    model="qwen/qwen3-32b",
    groq_api_key=API_KEY_GROQ,
    temperature=0.1,  # low creativity - fact based answer
    max_tokens=1024   # maximum tokens for generated response
)

In [49]:
# generate our retrieval-augmented output
def generate_output(query, retriever, llm, top_k=3):
    results = retriever.retrieve(query, top_k)
    
    context = "\n".join(doc["document"] for doc in results) if results else ""
    
    if not context:
        print("We found no relevant context for the given query")
        
        
    # prompt = context + query
    prompt = f""" use given context to generate the answer
                Context: {context}
                Query: {query} """
                
    response = llm.invoke(prompt.format(context=context, query=query)) # groq expects a list
    return response.content

In [50]:
answer = generate_output("What is RAG?", rag_retriever, llm)

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

embedding shape (1, 384)


In [51]:
print(answer)

<think>
Okay, the user is asking, "What is RAG?" Let me start by recalling the context provided. The context mentions a survey paper that reviews state-of-the-art RAG methods, their evolution through paradigms like naive RAG, and effective RAG frameworks. It also talks about evaluation methods, tasks, datasets, and future directions.

First, I need to define RAG. From the context, RAG stands for Retrieval-Augmented Generation. The paper probably discusses how RAG combines retrieval of information from external sources with a generative model, like an LLM. The main idea is that instead of relying solely on the model's internal knowledge, it retrieves relevant documents and uses them to generate more accurate or up-to-date responses.

The context mentions different paradigms of RAG, such as naive RAG and effective RAG. Naive RAG might be the initial approach where retrieval is done without much optimization, while effective RAG could involve more sophisticated techniques for better perfo

#### Claude - Anthropic

In [ ]:
# %pip install langchain-anthropic